# Train the prostate lesion model

This notebook clones the source code into Colab's local storage, extracts the separately archived processed splits locally for fast loading, and saves model checkpoints to Google Drive.

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from urllib.parse import urlparse, unquote
from google.colab import output


automatic branch detection

In [ ]:
project_dir = Path('/content/Masters_quad_joined_prostate_detection')
repository_url = (
    'https://github.com/Peter-worm/'
    'Masters_quad_joined_prostate_detection.git'
)

def run_git(args, capture_output=False):
    return subprocess.run(
        ['git'] + args,
        check=True,
        text=True,
        capture_output=capture_output,
    )

# 1. Detect the source notebook URL
try:
    notebook_url = output.eval_js('window.top.location.href')
except Exception:
    notebook_url = ''

if not notebook_url or '/blob/' not in urlparse(notebook_url).path:
    notebook_url = input(
        'Paste the original GitHub-backed Colab notebook URL: '
    ).strip()

parsed = urlparse(notebook_url)
parts = [unquote(p) for p in parsed.path.split('/') if p]

# 2. Extract and validate the repository
if parsed.hostname == 'colab.research.google.com':
    if len(parts) < 4 or parts[0] != 'github':
        raise ValueError('Expected a GitHub-backed Colab URL.')
    owner, repo = parts[1], parts[2]

elif parsed.hostname in ('github.com', 'www.github.com'):
    if len(parts) < 4:
        raise ValueError('Invalid GitHub URL.')
    owner, repo = parts[0], parts[1]

else:
    raise ValueError('Unsupported URL. Provide the original GitHub URL.')

expected_repo = urlparse(repository_url).path.strip('/')
expected_repo = expected_repo.removesuffix('.git')

if f'{owner}/{repo}'.lower() != expected_repo.lower():
    raise ValueError(
        f'Notebook URL points to {owner}/{repo}, '
        f'not {expected_repo}.'
    )

if 'blob' not in parts:
    raise ValueError('Cannot find the notebook branch in the URL.')

blob_index = parts.index('blob')
path_after_blob = '/'.join(parts[blob_index + 1:])

# 3. Match the URL against actual remote branches
result = run_git(
    ['ls-remote', '--heads', repository_url],
    capture_output=True,
)

remote_branches = [
    line.split('\t', 1)[1].removeprefix('refs/heads/')
    for line in result.stdout.splitlines()
    if '\trefs/heads/' in line
]

matches = [
    branch for branch in remote_branches
    if path_after_blob.startswith(branch + '/')
]

if not matches:
    raise ValueError(
        'Could not identify the source branch. '
        'Check the notebook URL and repository access.'
    )

branch_name = max(matches, key=len)

print(f'Repository: {owner}/{repo}')
print(f'Detected branch: {branch_name}')
print(f'Notebook file: {path_after_blob[len(branch_name) + 1:]}')

# 4. Clone or update the existing checkout
if not project_dir.exists():
    run_git([
        'clone', '--depth', '1',
        '--single-branch',
        '--branch', branch_name,
        repository_url, str(project_dir),
    ])

elif not (project_dir / '.git').is_dir():
    raise FileExistsError(
        f'{project_dir} exists but is not a Git repository.'
    )

else:
    # Confirm that the existing checkout belongs to this repository.
    remote = run_git(
        ['-C', str(project_dir), 'remote', 'get-url', 'origin'],
        capture_output=True,
    ).stdout.strip()

    normalize = lambda url: url.rstrip('/').removesuffix('.git').lower()

    if normalize(remote) != normalize(repository_url):
        raise ValueError(
            f'Existing checkout has a different origin: {remote}'
        )

    # Fetch the detected branch.
    run_git([
        '-C', str(project_dir), 'fetch', 'origin',
        f'refs/heads/{branch_name}:refs/remotes/origin/{branch_name}',
    ])

    local_branch = subprocess.run([
        'git', '-C', str(project_dir),
        'show-ref', '--verify', '--quiet',
        f'refs/heads/{branch_name}',
    ]).returncode == 0

    if local_branch:
        run_git(['-C', str(project_dir), 'switch', branch_name])
        run_git([
            '-C', str(project_dir), 'merge', '--ff-only',
            f'origin/{branch_name}',
        ])
    else:
        run_git([
            '-C', str(project_dir), 'switch', '--track',
            '-c', branch_name, f'origin/{branch_name}',
        ])

os.chdir(project_dir)

print(f'Project directory: {project_dir}')
print(f'Current branch: {subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()}')

## Optional: update the local checkout

Run the next cell after pushing code changes to GitHub and before importing project modules. Set `UPDATE_REPOSITORY` to `False` when you want to keep the current checkout.

In [ ]:
UPDATE_REPOSITORY = True

if UPDATE_REPOSITORY and (project_dir / '.git').is_dir():
    update_result = subprocess.run(
        ['git', '-C', str(project_dir), 'pull', '--ff-only'],
        capture_output=True,
        text=True,
    )
    if update_result.returncode != 0:
        raise RuntimeError(
            'Unable to update the local checkout. Commit or discard local '
            f'changes, then retry:\n{update_result.stderr.strip()}'
        )
    print(update_result.stdout.strip() or 'Repository is already up to date.')


In [ ]:
%pip install -r requirements.txt


In [ ]:
drive_archive_dir = Path('/content/drive/MyDrive/Master_Thesis/processed_archives')
local_processed_dir = project_dir / 'data' / 'processed'
weights_dir = Path('/content/drive/MyDrive/Master_Thesis/training_checkpoints')

if local_processed_dir.exists():
    shutil.rmtree(local_processed_dir)
local_processed_dir.mkdir(parents=True, exist_ok=True)

for split in ('train', 'test'):
    archive_path = drive_archive_dir / f'processed_{split}.tar.gz'
    if not archive_path.is_file():
        raise FileNotFoundError(
            f'Missing processed archive: {archive_path}. '
            'Run 01_preprocessing.ipynb first.'
        )
    split_dir = local_processed_dir / split
    split_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as archive:
        archive.extractall(split_dir)
    extracted_files = list(split_dir.glob('*.pkl'))
    if not extracted_files:
        raise RuntimeError(f'Archive contains no pickle files: {archive_path}')
    print(f'Extracted {len(extracted_files)} files to {split_dir}')
weights_dir.mkdir(parents=True, exist_ok=True)
os.environ['AI_PROJECT_DIR'] = str(project_dir)
os.environ['SAVED_WEIGHTS_DIR'] = str(weights_dir)
sys.path.insert(0, str(project_dir))


## Start training

The current training implementation reports the `test` split as validation during training. Treat the final test score accordingly.

In [ ]:
from src.models.train import train

train()
